In [1]:
!pip install -q jq langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
from huggingface_hub import login
# Pega tu token cuando aparezca el cuadro de texto
login()

In [4]:
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import json

# 1. Verificamos el archivo primero (solo para estar seguros)
with open('ashaninka_hf.jsonl', 'r', encoding='utf-8') as f:
    primera_linea = json.loads(f.readline())
    print("Estructura detectada:", primera_linea)

# 2. Función de metadatos ultra-robusta
def metadata_func(record: dict, metadata: dict) -> dict:
    # Intentamos sacar la info esté donde esté
    trans = record.get("translation", record) # Si no está en 'translation', busca en la raíz
    metadata["es"] = trans.get("es", "vacio")
    metadata["cni"] = trans.get("cni", "vacio")
    return metadata

# 3. Cargador corregido
loader = JSONLoader(
    file_path='ashaninka_hf.jsonl',
    jq_schema='.translation' if 'translation' in primera_linea else '.',
    content_key='es',
    metadata_func=metadata_func,
    json_lines=True
)

# 4. Crear el buscador
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vector_db = FAISS.from_documents(loader.load(), embeddings)

print("✅ Buscador reconstruido. Si arriba dice 'Estructura detectada', vamos bien.")

Estructura detectada: {'translation': {'es': '¿está?', 'cni': 'saikatsi'}}


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Buscador reconstruido. Si arriba dice 'Estructura detectada', vamos bien.


In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

model_id = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

# Configuración para que el modelo quepa en la GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("🚀 Modelo cargado y listo para traducir.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

🚀 Modelo cargado y listo para traducir.


In [13]:
#Finalista

# 1. FUNCIÓN DE APOYO: Extraer nombres
def extraer_nombres_propios(frase):
    palabras = frase.split()
    excluir_inicio = ["En", "El", "La", "Los", "Las", "Un", "Una", "Vamos", "Hoy"]
    nombres = []
    for i, p in enumerate(palabras):
        p_limpia = p.strip(",.¿?¡!")
        if len(p_limpia) > 0 and p_limpia[0].isupper():
            if i == 0 and p_limpia in excluir_inicio:
                continue
            nombres.append(p_limpia)
    return list(set(nombres))

# 2. FUNCIÓN DE APOYO: Simplificación semántica
def simplificar_para_diccionario(frase_usuario):
    prompt_simples = f"<s>[INST] Simplifica a palabras clave (sustantivos y verbos en infinitivo): {frase_usuario} [/INST] Palabras clave:"
    res = pipe(prompt_simples, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return res[0]['generated_text'].split("clave:")[-1].strip().lower()

def traductor_ashaninka_paso_a_paso(frase_usuario):
    # 1. Identificar nombres
    nombres_protegidos = extraer_nombres_propios(frase_usuario)

    # 2. Simplificación Semántica
    conceptos = simplificar_para_diccionario(frase_usuario)

    # 3. Diccionario de Control (Palabras clave que faltan en tu RAG)
    gramatica_base = {
        "vamos": "tsame", "estudiar": "yotaantsi", "aprender": "yotaantsi",
        "escuela": "yotaantsipanko", "a": "anta", "la": "oka"
    }

    # 4. RAG Dinámico con Filtro de Relevancia
    palabras = frase_usuario.lower().replace("¿", "").replace("?", "").split()
    contexto_lista = []

    for p in palabras:
        if p in gramatica_base:
            contexto_lista.append(f"{p} = {gramatica_base[p]}")

        if p.capitalize() not in nombres_protegidos and len(p) > 2:
            # k=3 para dar un poco más de contexto semántico
            docs = vector_db.similarity_search_with_score(p, k=3)
            for doc, score in docs:
                es_val = doc.metadata.get('es', '').lower()
                cni_val = doc.metadata.get('cni', '')
                if score < 0.85: # Umbral balanceado
                    contexto_lista.append(f"{es_val} = {cni_val}")

    contexto = "\n".join(list(set(contexto_lista)))
    print(f"DEBUG - Diccionario Filtrado:\n{contexto if contexto else 'VACÍO'}")

    # 5. PROMPT ANTI-NOTAS (Instrucción directa para evitar inglés)
    prompt_final = f"""<s>[INST] Eres un traductor de Español a Asháninka.
Usa SOLO el DICCIONARIO.
PROHIBIDO escribir en inglés. PROHIBIDO dar explicaciones.
Solo escribe la frase traducida.

DICCIONARIO:
{contexto}

FRASE: {frase_usuario} [/INST] """

    # 6. GENERACIÓN
    res_final = pipe(
        prompt_final,
        max_new_tokens=40,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # 7. LIMPIEZA TOTAL (Elimina paréntesis y texto extra)
    salida = res_final[0]['generated_text'].split("[/INST]")[-1].strip()
    # Limpiamos cualquier nota que la IA haya puesto entre paréntesis
    resultado = salida.split('\n')[0].split('(')[0].replace("Asháninka:", "").strip()

    return resultado

# --- PRUEBA ---
print(f"\nResultado final: {traductor_ashaninka_paso_a_paso('vamos a estudiar a la escuela')}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'pad_token_id', 'do_sample', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=40) 

DEBUG - Diccionario Filtrado:
escuela = yotaantsipanko
vamos = tsame
la = oka
estudiar = yotaantsi
a = anta

Resultado final: Tsame anta yotaantsipanko yotaantsi oka.


In [16]:
def iniciar_interfaz_traductor():
    print("="*50)
    print("✨ TRADUCTOR ASHÁNINKA INTERACTIVO (TESIS V7.0) ✨")
    print("Escribe la frase que deseas traducir.")
    print("Para salir, escribe 'exit', 'salir' o 'quit'.")
    print("="*50)

    while True:
        # 1. Capturar la entrada del usuario
        entrada = input("\n📝 Frase en Español: ").strip()

        # 2. Condición de salida
        if entrada.lower() in ['exit', 'salir', 'quit']:
            print("\n👋 ¡Añayitajimpari! (Gracias). Cerrando el traductor...")
            break

        # 3. Validar que no esté vacío
        if not entrada:
            continue

        try:
            # 4. Llamar a tu función principal
            # Asegúrate de que la función 'traductor_ashaninka_paso_a_paso' esté definida arriba
            resultado = traductor_ashaninka_paso_a_paso(entrada)

            print(f"🌿 Asháninka: {resultado}")

        except Exception as e:
            print(f"❌ Ocurrió un error: {e}")

# --- EJECUTAR EL TRADUCTOR ---
iniciar_interfaz_traductor()

✨ TRADUCTOR ASHÁNINKA INTERACTIVO (TESIS V7.0) ✨
Escribe la frase que deseas traducir.
Para salir, escribe 'exit', 'salir' o 'quit'.


Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DEBUG - Diccionario Filtrado:
río = tampo
la = oka
peces = shimayetatsiripe
río = ene
mañana = osaitekera
vamos = tsame
a = anta
🌿 Asháninka: tsame anta shimayetatsiripe ene osaitekera


Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DEBUG - Diccionario Filtrado:
caracol = shebori
hermana = aari
hermana = tsionti
cocina = okotsiti
hermana = choki
caracol = sankiro
🌿 Asháninka: aari okotsiti shebori


Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DEBUG - Diccionario Filtrado:
loro = eroti
loro = chorito
loro = kintaro
lleva = inkenanake
enfermo = jokiitaka
enfermo = jokiirenti
🌿 Asháninka: mi kintaro jokiitaka

📝 Frase en Español: exit

👋 ¡Añayitajimpari! (Gracias). Cerrando el traductor...
